In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots

df = load_all_snapshots()

rel = (
    df[df["pitch_type"].notna()]
    .groupby("p_throws")
    .agg(
        n=("release_pos_x", "size"),
        x_mean=("release_pos_x", "mean"),
        x_std=("release_pos_x", "std"),
        z_mean=("release_pos_z", "mean"),
        z_std=("release_pos_z", "std"),
        ext_mean=("release_extension", "mean"),
    )
    .round(2)
)
print(rel.to_string())

               n  x_mean  x_std  z_mean  z_std  ext_mean
p_throws                                                
L         190100    2.01    0.8     5.8   0.52      6.41
R         517865   -1.87    0.7    5.74   0.53      6.47


In [2]:
work = df[df["pitch_type"].notna()].copy()
work["rel_x_arm"] = np.where(work["p_throws"] == "L",
                              -work["release_pos_x"], work["release_pos_x"])

MIN_TOTAL = 500
totals = work.groupby("pitcher").size()
eligible = totals[totals >= MIN_TOTAL].index
w = work[work["pitcher"].isin(eligible)]

consistency = (
    w.groupby("pitcher")
    .agg(
        n=("rel_x_arm", "size"),
        x_std=("rel_x_arm", "std"),
        z_std=("release_pos_z", "std"),
        ext_std=("release_extension", "std"),
    )
)
consistency["release_spread"] = np.hypot(consistency["x_std"], consistency["z_std"])
print(consistency[["x_std", "z_std", "release_spread"]].describe().round(3).to_string())

         x_std  z_std  release_spread
count  473.000  473.0           473.0
mean     0.215   0.14            0.26
std      0.097  0.037           0.095
min      0.090  0.077            0.14
25%      0.157  0.115             0.2
50%      0.187  0.133           0.233
75%      0.240  0.158            0.29
max      0.813  0.308           0.829


In [3]:
by_type = (
    w.groupby(["pitcher", "pitch_type"])
    .agg(n=("rel_x_arm", "size"),
         x_mean=("rel_x_arm", "mean"),
         z_mean=("release_pos_z", "mean"),
         x_std=("rel_x_arm", "std"),
         z_std=("release_pos_z", "std"))
)
by_type = by_type[by_type["n"] >= 50]

def release_tell(pitcher_id):
    """How far apart are this pitcher's release points BETWEEN pitch types,
    relative to the scatter WITHIN each type?"""
    a = by_type.loc[pitcher_id]
    if len(a) < 2:
        return None

    # spread of pitch-type centres
    between = np.hypot(a["x_mean"].std(), a["z_mean"].std())
    # typical scatter within a pitch type
    within = np.hypot(a["x_std"].mean(), a["z_std"].mean())
    return pd.Series({"between": between, "within": within,
                      "ratio": between / within if within else np.nan})

tells = pd.DataFrame({
    pid: release_tell(pid)
    for pid in by_type.index.get_level_values(0).unique()
}).T.dropna()

print(tells.describe().round(3).to_string())

       between   within    ratio
count  473.000  473.000  473.000
mean     0.146    0.223    0.690
std      0.077    0.085    0.343
min      0.017    0.124    0.094
25%      0.093    0.172    0.454
50%      0.133    0.197    0.612
75%      0.180    0.243    0.891
max      0.733    0.771    2.774


In [4]:
from src.data.player_ids import load_player_ids, display_name

ids = load_player_ids(tells.index.tolist())
names = display_name(ids)
tells["name"] = tells.index.map(names)

print("=== most consistent release (lowest ratio) ===")
print(tells.nsmallest(10, "ratio")[["name", "between", "within", "ratio"]].round(3).to_string())
print()
print("=== most variable release (highest ratio) ===")
print(tells.nlargest(10, "ratio")[["name", "between", "within", "ratio"]].round(3).to_string())

=== most consistent release (lowest ratio) ===
                       name  between  within  ratio
686539       Cronin, Declan    0.017   0.187  0.094
664854        Helsley, Ryan    0.030   0.252  0.120
628452     Iglesias, Raisel    0.068   0.449  0.151
608638      Chargois, J. T.    0.029   0.185  0.156
489446         Yates, Kirby    0.031   0.185  0.169
502085     Robertson, David    0.031   0.181  0.170
642546  Hernández, Jonathan    0.034   0.194  0.177
605154        Brebbia, John    0.040   0.229  0.177
643338          Green, Chad    0.052   0.286  0.182
571946       Miller, Shelby    0.042   0.225  0.185

=== most variable release (highest ratio) ===
                    name  between  within  ratio
669062      Miller, Erik    0.513   0.185  2.774
605397     Musgrove, Joe    0.263   0.141  1.861
542881   Anderson, Tyler    0.733   0.397  1.845
592767       Smyly, Drew    0.253   0.154  1.649
622253      Tate, Dillon    0.309   0.190  1.629
687765     Spence, Mitch    0.280   0.17

In [5]:
from src.features.plate_discipline import add_swing_flags

flagged = add_swing_flags(w)
whiff_by_pitcher = (
    flagged[flagged["is_swing"]]
    .groupby("pitcher")["is_whiff"]
    .agg(["mean", "size"])
)
whiff_by_pitcher = whiff_by_pitcher[whiff_by_pitcher["size"] >= 200]

joined = tells.join(whiff_by_pitcher).dropna(subset=["mean"])
print(f"{len(joined)} pitchers")
print("ratio vs whiff rate:  ", round(joined["ratio"].corr(joined["mean"]), 3))
print("between vs whiff rate:", round(joined["between"].corr(joined["mean"]), 3))
print("within vs whiff rate: ", round(joined["within"].corr(joined["mean"]), 3))

473 pitchers
ratio vs whiff rate:   0.018
between vs whiff rate: 0.014
within vs whiff rate:  0.007


In [6]:
from src.features.arsenal import build_arsenal, arsenal_depth

ars = build_arsenal(w)
depth = arsenal_depth(ars, min_total=500)
tells_d = tells.join(depth["n_pitch_types"])

print("ratio vs arsenal size:", round(tells_d["ratio"].corr(tells_d["n_pitch_types"]), 3))
print("between vs arsenal size:", round(tells_d["between"].corr(tells_d["n_pitch_types"]), 3))
print()
print(tells_d.groupby("n_pitch_types")["ratio"].agg(["mean", "size"]).round(3).to_string())

ratio vs arsenal size: 0.059
between vs arsenal size: 0.077

                mean  size
n_pitch_types             
2              0.624    44
3              0.686   122
4              0.686   154
5              0.714   101
6              0.727    44
7              0.640     5
8              0.727     2
9              0.659     1
